# PBL01_??????????????????

???????????PBL01_???????????????AI??(???)???????????????????????????????????????????????????????????????????????????????????????????????

????????????????????????????

1. ????????
2. ???????????
3. ???
4. ?????
5. ?????
6. ??????????

?????????????????????????????????????????????

# ???????????

In [ ]:
from itertools import product

import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.linear_model import LinearRegression

# 1. ????????: pandas.read_csv()

In [ ]:
sales = pd.read_csv('./inputs/sales_history.csv')
cats = pd.read_csv('./inputs/item_categories.csv')
cat_names = pd.read_csv('./inputs/category_names.csv')
test = pd.read_csv('./inputs/test.csv')
sub = pd.read_csv('./inputs/sample_submission.csv', header=None)

### ?????????

In [ ]:
# sales_history.csv?????????
print(sales.info())
sales.head()

In [ ]:
# item_categories.csv?????????
print(cats.info())
cats.head()

In [ ]:
# cat_names?????????
print(cat_names.info())
cat_names.head()

In [ ]:
# test.csv?????????
print(test.info())
test.head()

In [ ]:
# sample_submission.csv?????????
print(sub.info())
sub.head()

# 2. ???????????

### ??ID??????: pandas.Series.nunique()

In [ ]:
# train??????????ID????
print(sales['??ID'].nunique())

In [ ]:
# test??????????ID????
print(test['??ID'].nunique())

### ???????(??): pd.merge(df1, df2)

?sales_history.csv???item_category.csv???category_names.csv??3????????????

In [ ]:
merged_data = pd.merge(sales, cats, on='??ID', how='left')
merged_data = pd.merge(merged_data, cat_names, on='??????ID', how='left')

merged_data.head()

???ID?????????ID???????????????????????1?????????????????????

### ????????????????????????????: pandas.Series.apply(), str.split()

??????????????????????????????????

In [ ]:
# ????????????????????????????
sales['?'] = sales['??'].apply(lambda x: x.split('-')[0])
sales['?'] = sales['??'].apply(lambda x: x.split('-')[1])

sales.head()

In [ ]:
# ???????????????????
gp_time = sales.groupby(['?', '?']).count().reset_index()[['?', '?']]
gp_time

??????????????2018?1??2019?10???22??????????????<br>
??????????????????????????????????????????

- 2018?1?: `0`
- 2018?2?: `1`

...

- 2019?9?: `20`
- 2019?10?: `21`

In [ ]:
# ???????????????????????
gp_time['?????'] = list(range(len(gp_time)))

# test???????????????2019?12?(?????: 23)??????????
gp_time = gp_time.append({
    '?????': 23,
    '?': '2019',
    '?': '12'
}, ignore_index=True)

gp_time

In [ ]:
# ???????????sales?????
sales = pd.merge(sales, gp_time, on=['?', '?'], how='left')
sales

### ?????????????(????????): pandas.DataFrame.groupby()

???????????????????????????????????

In [ ]:
sales_month = sales.groupby('?????').agg({'????': np.sum}).reset_index()
sales_month.head()

In [ ]:
plt.bar(sales_month['?????'], sales_month['????'])

12?(?????:11)???????????????????????

# 3. ???

### ???????????(???ID?????ID?)

?????????????????????????????????<br>
???????????????????????2019?12??1?????????????????????<br>
????????????????????????????????????

In [ ]:
# ??????????ID????ID????????????????????
gp_month = sales.groupby(['?????',  '??ID', '??ID']).agg({'????': np.sum}).reset_index()
gp_month

### ?'?'?'??ID'? '??ID'?????????????

sales?????????0????????????????<br>
?????????????0????????????????????????????????????<br>
????????????ID?????ID??????????????????????????????0??????????????????????????

In [ ]:
all_combination = []

# '?'?'??ID'? '??ID'????????????
columns = ['?????', '??ID', '??ID']

for i in range(22):
    train_block = sales[sales['?????']==i]
    all_combination.append(np.array(list(product([i], train_block['??ID'].unique(), train_block['??ID'].unique()))))
    
all_combination = pd.DataFrame(np.vstack(all_combination), columns=columns)
all_combination.sort_values(columns, inplace=True)

print(all_combination.info())
all_combination

In [ ]:
# all_combination?????????????
all_train = pd.merge(all_combination, gp_month, on=['?????', '??ID', '??ID'], how='left')

# ?????NaN??????????????0?????????
all_train['????'] = all_train['????'].fillna(0)

all_train

### test????????????

In [ ]:
new_test = test.copy()

# ???????2019?12?(?????: 23)???????????
new_test['?????'] = 23

new_test.head()

### ?????????????????: pd.concat()

In [ ]:
dataset = pd.concat([all_train, new_test])
dataset

In [ ]:
dataset.?????.unique()

????????????23(2019?12?)????????????????????NaN???

# 4. ?????
- <font color="#FF0000"><strong>?????????????????????????????????????????????????????????????????????</strong></font>
- ?????1?????????????????????

### ????????

????????????????????????????????????????????????????<br>
???????????1??(12)???????????????????????

In [ ]:
lag = dataset.copy()
    
lag['?????'] = lag['?????'] + 12
lag = lag.rename(columns={'????': '12????????'})
lag

### ????????

In [ ]:
dataset2 = pd.merge(dataset, lag, on=['?????', '??ID', '??ID'], how='left', left_index=True)
print(dataset2.groupby('?????').agg({'12????????': 'count'}))

? ???????0?11??????????????12?????????????????????12??????????????????

### ??????????????????

In [ ]:
dataset3 = pd.merge(dataset2, cats, on='??ID', how='left')
dataset3 = pd.merge(dataset3, gp_time, on='?????', how='left')

print(dataset3.info())
dataset3.head()

### ???????????????: pandas.Series.astype()

In [ ]:
dataset3['?'] = dataset3['?'].astype(int)
dataset3['?'] = dataset3['?'].astype(int)

### ????0????

In [ ]:
dataset3 = dataset3.fillna(0)
dataset3.head()

## ?????????

### ????????????????????

In [ ]:
# ???????????12????????????????????????????
train_dataset = dataset3[(dataset3['?????'] >= 12) & (dataset3['?????'] != 23)]

test_dataset = dataset3[dataset3['?????'] == 23]

In [ ]:
train_dataset

In [ ]:
test_dataset

### ?????????????????

In [ ]:
X_train = train_dataset.drop('????', axis=1)
y_train = train_dataset['????']

X_test = test_dataset.drop(['????'], axis=1)

# 5. ?????
- <font color="#FF0000"><strong>?????????????????????????????????????????????????????????</strong></font>
- ????????????????????????????????????????????????

In [ ]:
# ????????????
model = LinearRegression()

# ??????
model.fit(X_train, y_train)

In [ ]:
# ???????????????????????????????
y_pred = model.predict(X_test)

y_pred

# 6. ??????????: pandas.DataFrame.to_csv()

sample_submission.csv????????????????

In [ ]:
# sample_submission??????
sub.head()

In [ ]:
# sample_submission?????????????????
sub.iloc[:, -1] = y_pred

sub.head()

In [ ]:
# ?????????
sub.to_csv('my_submission.csv', index=False, header=False)